In [ ]:
UPDATE = False
PresentDate = "2022-06-01"

In [ ]:
from yahoo_fin import options
# specific expiration date
aa = options.get_options_chain("S&P500", "04/26/2019")

In [ ]:
import numpy as np
import pandas as pd
from nltk.classify import NaiveBayesClassifier
from nltk.corpus import subjectivity
from nltk.sentiment import SentimentAnalyzer
from nltk.sentiment.util import *

#pd.set_option('display.max_colwidth', -1)
from time import time
import re
import string
import os
from pprint import pprint
import collections

import matplotlib.pyplot as plt
import seaborn as sns
sns.set(style="darkgrid")
sns.set(font_scale=1.3)

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.metrics import classification_report

from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
#from sklearn.externals import joblib
import joblib as jb

#import gensim

from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from nltk.tokenize import word_tokenize

import warnings
warnings.filterwarnings('ignore')

np.random.seed(37)

SHOW_GRAPHS = True

In [ ]:
df  = pd.DataFrame()

df_companies  = pd.DataFrame()
df_time  = pd.DataFrame()
df_hour  = pd.DataFrame()

df_companies=pd.read_pickle('/Users/Farid Srouji/Documents/Research/master/financialmodelingprep/data/pkl/SPXCompaniesByMarketCap.pkl')

In [ ]:
df_companies#["MarketCapMillions"]

In [ ]:
df_companies.index["symbol"]

In [ ]:
# df_companies.groupby("Industry").count().sort_values(by=['Rank'])

In [ ]:
df_articles = pd.read_pickle('/Users/Farid Srouji/Documents/Research/master/financialmodelingprep/data/pkl/df_dates.pkl', compression=None)

In [ ]:
df_companies.rename(columns={"symbol":"Company"})

In [ ]:
df_companies.remove_duplicate()

In [ ]:
df_companies.merge(df_articles,on="date")

## Sentiment

In [ ]:
# Reading the saved data pickle file

df_sentiment  = pd.DataFrame()

df_sentiment=pd.read_pickle('/Users/Farid Srouji/Documents/Research/master/financialmodelingprep/data/pkl/SentimentTime.pkl', compression=None)
# df_sentiment.drop(columns=["compound", "neg", "pos", "neu", "Articles", "Title", "Text", "Company"], inplace=True)
df_sentiment.drop(columns=["Company"], inplace=True)
df_sentiment.index = pd.to_datetime(df_sentiment.index)
df_sentiment.Date = pd.to_datetime(df_sentiment.Date)
# df_sentiment = df_sentiment[df_sentiment["sentiment"]>0]

df_sentiment = df_sentiment.ffill()

df_sentiment["sentiment(t-1)"] = df_sentiment["sentiment"].shift(periods=1)
df_sentiment["sentiment(t-2)"] = df_sentiment["sentiment"].shift(periods=2)
df_sentiment["sentiment(t-1)"] = df_sentiment["sentiment(t-1)"].ffill()
df_sentiment["sentiment(t-2)"] = df_sentiment["sentiment(t-2)"].ffill()

# SENTIMENT RETURN
df_sentiment["sentiment_return"]=(df_sentiment["sentiment(t-1)"]-df_sentiment["sentiment(t-2)"])/df_sentiment["sentiment(t-2)"]
# SENTIMENT LN RETURN
df_sentiment["sentiment_ln_return"]=np.log(df_sentiment["sentiment(t-1)"]/df_sentiment["sentiment(t-2)"])
# SENTIMENT EWM
df_sentiment["sentiment_ewm10"]=df_sentiment["sentiment"].ewm(10).mean()
df_sentiment["sentiment_ewm20"]=df_sentiment["sentiment"].ewm(20).mean()

# POSITIVE LN RETURN
df_sentiment["pos(t-1)"] = df_sentiment["pos"].shift(periods=1)
df_sentiment["pos(t-2)"] = df_sentiment["pos"].shift(periods=2)
df_sentiment["pos(t-1)"] = df_sentiment["pos(t-1)"].ffill()
df_sentiment["pos(t-2)"] = df_sentiment["pos(t-2)"].ffill()
df_sentiment["pos_ln_return"]=np.log(df_sentiment["pos(t-1)"]/df_sentiment["pos(t-2)"])

# NEGATIVE LN RETURN
df_sentiment["neg(t-1)"] = df_sentiment["neg"].shift(periods=1)
df_sentiment["neg(t-2)"] = df_sentiment["neg"].shift(periods=2)
df_sentiment["neg(t-1)"] = df_sentiment["neg(t-1)"].ffill()
df_sentiment["neg(t-2)"] = df_sentiment["neg(t-2)"].ffill()
df_sentiment["neg_ln_return"]=np.log(df_sentiment["neg(t-1)"]/df_sentiment["neg(t-2)"])

# NEUTRAL LN RETURN
df_sentiment["neu(t-1)"] = df_sentiment["neu"].shift(periods=1)
df_sentiment["neu(t-2)"] = df_sentiment["neu"].shift(periods=2)
df_sentiment["neu(t-1)"] = df_sentiment["neu(t-1)"].ffill()
df_sentiment["neu(t-2)"] = df_sentiment["neu(t-2)"].ffill()
df_sentiment["neu_ln_return"]=np.log(df_sentiment["neu(t-1)"]/df_sentiment["neu(t-2)"])

# COMPOUND LN RETURN
df_sentiment["compound(t-1)"] = df_sentiment["compound"].shift(periods=1)
df_sentiment["compound(t-2)"] = df_sentiment["compound"].shift(periods=2)
df_sentiment["compound(t-1)"] = df_sentiment["compound(t-1)"].ffill()
df_sentiment["compound(t-2)"] = df_sentiment["compound(t-2)"].ffill()
df_sentiment["compound_ln_return"]=np.log(df_sentiment["compound(t-1)"]/df_sentiment["compound(t-2)"])

df_sentiment['WordCount'] = df_sentiment['Articles'].apply(lambda x:len(str(x).split())) #Number Of words in main text
df_sentiment["WordCount(t-1)"] = df_sentiment["WordCount"].shift(periods=1)
df_sentiment["WordCount(t-2)"] = df_sentiment["WordCount"].shift(periods=2)
df_sentiment["WordCount(t-1)"] = df_sentiment["WordCount(t-1)"].ffill()
df_sentiment["WordCount(t-2)"] = df_sentiment["WordCount(t-2)"].ffill()
df_sentiment["WordCount_ln_return"]=np.log(df_sentiment["WordCount(t-1)"]/df_sentiment["WordCount(t-2)"])

In [ ]:
df = df_sentiment

# DAY MONTH YEAR

In [ ]:
import datetime as dt
from datetime import date
import calendar
my_date = date.today()
calendar.day_name[my_date.weekday()]  #'Wednesday'

# Prepare data
df['year'] = [d.year for d in df.Date]
df['month'] = [d.strftime('%b') for d in df.Date]
df['day'] = [d.day_name() for d in df.Date]
years = df['year'].unique()
df['month-year'] = [d.strftime('%b') + str(d.year) for d in df.Date]

# VIX

In [ ]:
if (UPDATE):
    from yahoo_fin.stock_info import get_data
    df_vix = get_data("^VIX", start_date="25/09/2019", end_date=PresentDate, index_as_date = True, interval="1d")
    df_vix.to_pickle("/Users/Farid Srouji/Documents/Research/master/financialmodelingprep/data/pkl/vix.pkl")

In [ ]:
df_vix=pd.read_pickle('/Users/Farid Srouji/Documents/Research/master/financialmodelingprep/data/pkl/vix.pkl', compression=None)

df_vix.index = pd.to_datetime(df_vix.index)
df_vix["Date"]= pd.to_datetime(df_vix.index)

df_vix["vix"] = df_vix["adjclose"]
# df_vix.drop(columns=["high", "low", "adjclose", "ticker", "volume"], inplace=True)
# df_vix["adjclose"]  = df_vix["adjclose"].ffill()

df_vix["adjclose(t-1)"] = df_vix["adjclose"].shift(periods=1)
df_vix["adjclose(t-2)"] = df_vix["adjclose"].shift(periods=2)

# vix RETURN
# df_vix["vix_return_c2c"]=(df_vix["vix"]-df_vix["vix_yesterday"])/df_vix["vix_yesterday"]
# df_vix["vix_return_c2c"]=(df_vix["close"]-df_vix["open"])/df_vix["open"]
# vix LN RETURN
# df_vix["vix_ln_return_c2c"]=np.log(df_vix["vix"]/df_vix["vix_yesterday"])
df_vix["vix_ln_return"]=np.log(df_vix["adjclose"]/df_vix["adjclose(t-1)"])
df_vix["vix_ln_return(t-1)"]=np.log(df_vix["adjclose(t-1)"]/df_vix["adjclose(t-2)"])

# vix EWM
df_vix["vix_ewm"]=df_vix["adjclose"].ewm(10).mean()

In [ ]:
if (UPDATE):
    from yahoo_fin.stock_info import get_data
    df_vix = get_data("^VIX", start_date="25/09/2019", end_date=PresentDate, index_as_date = True, interval="1d")
    df_vix.to_pickle("/Users/Farid Srouji/Documents/Research/master/financialmodelingprep/data/pkl/vix.pkl")

In [ ]:
df_vix=pd.read_pickle('/Users/Farid Srouji/Documents/Research/master/financialmodelingprep/data/pkl/vix.pkl', compression=None)

df_vix.index = pd.to_datetime(df_vix.index)
df_vix["Date"]= pd.to_datetime(df_vix.index)

df_vix["vix"] = df_vix["adjclose"]
df_vix.drop(columns=["high", "low", "adjclose", "ticker", "volume"], inplace=True)
# df_vix["adjclose"]  = df_vix["adjclose"].ffill()

df_vix["vix(t-1)"] = df_vix["vix"].shift(periods=1)
df_vix["vix(t-2)"] = df_vix["vix"].shift(periods=2)

# vix RETURN
# df_vix["vix_return_c2c"]=(df_vix["vix"]-df_vix["vix_yesterday"])/df_vix["vix_yesterday"]
# df_vix["vix_return_c2c"]=(df_vix["close"]-df_vix["open"])/df_vix["open"]
# vix LN RETURN
# df_vix["vix_ln_return_c2c"]=np.log(df_vix["vix"]/df_vix["vix_yesterday"])
df_vix["vix_ln_return"]=np.log(df_vix["vix"]/df_vix["vix(t-1)"])
df_vix["vix_ln_return(t-1)"]=np.log(df_vix["vix(t-1)"]/df_vix["vix(t-2)"])

# vix EWM
df_vix["vix_ewm"]=df_vix["vix"].ewm(10).mean()

# S&P500

In [ ]:
if (UPDATE):
    from yahoo_fin.stock_info import get_data
    df_sp500 = get_data("^GSPC", start_date="25/09/2019", end_date=PresentDate, index_as_date = True, interval="1d")
    df_sp500.to_pickle("/Users/Farid Srouji/Documents/Research/master/financialmodelingprep/data/pkl/sp500.pkl")

In [ ]:
df_sp500=pd.read_pickle('/Users/Farid Srouji/Documents/Research/master/financialmodelingprep/data/pkl/sp500.pkl', compression=None)

df_sp500.index = pd.to_datetime(df_sp500.index)
df_sp500["Date"]= pd.to_datetime(df_sp500.index)

df_sp500["sp500"] = df_sp500["adjclose"]
df_sp500.drop(columns=["open", "high", "low", "close", "adjclose", "ticker", "volume"], inplace=True)
df_sp500["sp500"]  = df_sp500["sp500"].ffill()

df_sp500["sp500_yesterday"] = df_sp500["sp500"].shift(periods=1)
# sp500 RETURN
df_sp500["sp500_return"]=(df_sp500["sp500"]-df_sp500["sp500_yesterday"])/df_sp500["sp500_yesterday"]
# sp500 LN RETURN
df_sp500["sp500_ln_return"]=np.log(df_sp500["sp500"]/df_sp500["sp500_yesterday"])
# sp500 EWM
df_sp500["sp500_ewm"]=df_sp500["sp500"].ewm(10).mean()

# MERGING 

In [ ]:
df_sentiment["Date"]= pd.to_datetime(df_sentiment.Date)

In [ ]:
df  = pd.DataFrame()
df = pd.merge(df_sentiment,df_vix, on='Date')
df = df.ffill()
df = df.sort_values(by="Date")

df = pd.merge(df, df_sp500, on='Date')#,how='outer'
df = df.ffill()
df = df.sort_values(by="Date")

# THREE PERIODS

In [ ]:
df.index = df.Date

In [ ]:
df_part1 = df.loc['2019-09-25':'2020-02-14']
df_part2 = df.loc['2020-02-15':'2020-05-31']
df_part3 = df.loc['2020-06-01':PresentDate]
# df_part3 = df_part3.ffill()
# df = df_part3
df_sentiment = df_sentiment.loc['2020-06-01':PresentDate]
df_vix = df_vix.loc['2020-06-01':PresentDate]

In [ ]:
mx = df_part3.loc[df_part3['sentiment_ln_return'] != np.inf, 'sentiment_ln_return'].max()
df_part3['sentiment_ln_return'].replace(np.inf,mx,inplace=True)
mn = df_part3.loc[df_part3['sentiment_ln_return'] != -np.inf, 'sentiment_ln_return'].min()
df_part3['sentiment_ln_return'].replace(-np.inf,mn,inplace=True)

mx = df_part3.loc[df_part3['neg_ln_return'] != np.inf, 'neg_ln_return'].max()
df_part3['neg_ln_return'].replace(np.inf,mx,inplace=True)
mn = df_part3.loc[df_part3['neg_ln_return'] != -np.inf, 'neg_ln_return'].min()
df_part3['neg_ln_return'].replace(-np.inf,mn,inplace=True)

mx = df_part3.loc[df_part3['neu_ln_return'] != np.inf, 'neu_ln_return'].max()
df_part3['neu_ln_return'].replace(np.inf,mx,inplace=True)
mn = df_part3.loc[df_part3['neu_ln_return'] != -np.inf, 'neu_ln_return'].min()
df_part3['neu_ln_return'].replace(-np.inf,mn,inplace=True)

mx = df_part3.loc[df_part3['compound_ln_return'] != np.inf, 'compound_ln_return'].max()
df_part3['compound_ln_return'].replace(np.inf,mx,inplace=True)
mn = df_part3.loc[df_part3['compound_ln_return'] != -np.inf, 'compound_ln_return'].min()
df_part3['compound_ln_return'].replace(-np.inf,mn,inplace=True)

mx = df_part3.loc[df_part3['pos_ln_return'] != np.inf, 'pos_ln_return'].max()
df_part3['pos_ln_return'].replace(np.inf,mx,inplace=True)
mn = df_part3.loc[df_part3['pos_ln_return'] != -np.inf, 'pos_ln_return'].min()
df_part3['pos_ln_return'].replace(-np.inf,mn,inplace=True)

mx = df_part3.loc[df_part3['neg_ln_return'] != np.inf, 'neg_ln_return'].max()
df_part3['neg_ln_return'].replace(np.inf,mx,inplace=True)
mn = df_part3.loc[df_part3['neg_ln_return'] != -np.inf, 'neg_ln_return'].min()
df_part3['neg_ln_return'].replace(-np.inf,mn,inplace=True)

mx = df_part3.loc[df_part3['neu_ln_return'] != np.inf, 'neu_ln_return'].max()
df_part3['neu_ln_return'].replace(np.inf,mx,inplace=True)
mn = df_part3.loc[df_part3['neu_ln_return'] != -np.inf, 'neu_ln_return'].min()
df_part3['neu_ln_return'].replace(-np.inf,mn,inplace=True)

In [ ]:
mn = df_part3.loc[df_part3['sentiment_ln_return'] != -np.inf, 'sentiment_ln_return'].min()
df_part3['sentiment_ln_return'] = df_part3['sentiment_ln_return'].fillna(mn)

mn = df_part3.loc[df_part3['vix_ln_return'] != -np.inf, 'vix_ln_return'].min()
df_part3['vix_ln_return'] = df_part3['vix_ln_return'].fillna(mn)

mn = df_part3.loc[df_part3['sp500_ln_return'] != -np.inf, 'sp500_ln_return'].min()
df_part3['sp500_ln_return'] = df_part3['sp500_ln_return'].fillna(mn)

In [ ]:
df_part3["ticker",MarketCap]